<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/SectorStrengthScore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade yfinance
!pip install  --upgrade pandas_ta
!pip install ta pandas_ta
!pip install scipy==1.16.2

Sector Score=w1(RS)+ w2(Momentum)+ w3(Breadth)+ w4(Trend) + w5(Efficiency)

Example weights (start here):
RS → 30%
Momentum → 25%
Breadth → 20%
Trend slope → 15%
Volatility efficiency → 10%

In [23]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime
from scipy.stats import linregress
# ensure reproducibility
import random
random.seed(42)
print("Libraries Installed!")

Libraries Installed!


# Short Term Sector Rotation Strategy

In [3]:
class STSectorRotationRanker:
    def __init__(self, price_data: pd.DataFrame, sector_etfs: list, benchmark: str = 'SPY'):
        """
        price_data: DataFrame of adjusted closes with Date index (ascending) and columns for tickers (including benchmark).
        sector_etfs: list of sector ETF tickers to rank (strings).
        benchmark: ticker string present in price_data used as market benchmark (default 'SPY').

        """
        self.price_data = price_data.copy().sort_index()
        self.sector_etfs = sector_etfs
        self.benchmark = benchmark

        if benchmark not in self.price_data.columns:
            raise ValueError(f"Benchmark {benchmark} not found in price_data columns.")

    def _relative_series(self, ticker):
        """Return ratio series ticker / benchmark (RS series)."""
        return self.price_data[ticker] / self.price_data[self.benchmark]

    def compute_basic_rs(self, short_window=21, mid_window=63, long_window=126):
        """Compute RS1M, RS3M and RS6M as relative % change vs benchmark over short/mid windows."""
        results = []
        for t in self.sector_etfs:
            if t not in self.price_data.columns:
                results.append({'ticker': t, 'RS1M': np.nan, 'RS3M': np.nan,'RS6M':np.nan})
                continue
            rel = self._relative_series(t)
            #rs1m = (rel.iloc[-1] / rel.shift(short_window).iloc[-1] - 1) if len(rel) > short_window else np.nan
            #rs3m = (rel.iloc[-1] / rel.shift(mid_window).iloc[-1] - 1) if len(rel) > mid_window else np.nan
            rs1m = rel.pct_change(short_window).iloc[-1] if len(rel) > short_window else np.nan
            rs3m = rel.pct_change(mid_window).iloc[-1] if len(rel) > mid_window else np.nan
            rs6m = rel.pct_change(long_window).iloc[-1] if len(rel) > long_window else np.nan
            results.append({'ticker': t, 'RS1M': rs1m, 'RS3M': rs3m,"RS6M":rs6m})
        return pd.DataFrame(results).set_index('ticker')

    def compute_rs_slope(self, window=21):
        """Compute linear regression slope of RS series over `window` bars (last window)."""
        slopes = {}
        x = np.arange(window)
        for t in self.sector_etfs:
            if t not in self.price_data.columns:
                slopes[t] = np.nan
                continue
            rel = self._relative_series(t).dropna()
            if len(rel) < window:
                slopes[t] = np.nan
                continue
            y = rel.values[-window:]
            slope, intercept, r, p, se = linregress(x, y)
            slopes[t] = slope
        return pd.Series(slopes, name='RS_Slope')

    def compute_rolling_rs_acceleration(self, slope_window=21, accel_window=5):
        """
        Compute acceleration = change in rolling slope over accel_window steps.
        slope_window: how many bars to fit each rolling slope
        accel_window: how many slope-steps to look back for acceleration
        """
        acc = {}
        for t in self.sector_etfs:
            if t not in self.price_data.columns:
                acc[t] = np.nan
                continue
            rel = self._relative_series(t).dropna()
            if len(rel) < slope_window + accel_window:
                acc[t] = np.nan
                continue
            slopes = []
            for i in range(len(rel) - slope_window + 1):
                y = rel.values[i:i+slope_window]
                x = np.arange(slope_window)
                slope = linregress(x, y)[0]
                slopes.append(slope)
            slopes = np.array(slopes)
            acc_val = slopes[-1] - slopes[-1-accel_window] if len(slopes) > accel_window else np.nan
            acc[t] = acc_val
        return pd.Series(acc, name='RS_Acceleration')

    def merge_with_breadth(self):
      """
      Merge RS factors with breadth metrics
      """
      rs = self.compute_basic_rs()
      slope = self.compute_rs_slope()
      accel = self.compute_rolling_rs_acceleration()
      merged_df = rs.join(slope).join(accel)
      return merged_df

    def compute_leader_score(self, rs1w=0.5, rs3w=0.3, rs6w=0.2,slopew=0, accelw=0):
        """
        rs1w=0.4, rs3w=0.4, slopew=0.15, accelw=0.05
        Produce a DataFrame with RS1M, RS3M, RS6M, RS_Slope, RS_Acceleration and Leader_Score.
        Leader_Score computed as weighted z-scores of metrics.
        """
        rs = self.compute_basic_rs()
        slope = self.compute_rs_slope()
        accel = self.compute_rolling_rs_acceleration()

        df = rs.join(slope).join(accel)

        def zscore_col(s):
            # Use standard zscore; safe-guard against zero std
            return (s - s.mean()) / (s.std(ddof=0) if s.std(ddof=0) != 0 else 1)

        z_rs1   = zscore_col(df['RS1M'])
        z_rs3   = zscore_col(df['RS3M'])
        z_rs6   = zscore_col(df['RS6M'])
        z_slope = zscore_col(df['RS_Slope'])
        z_accel = zscore_col(df['RS_Acceleration'])

        score = rs1w*z_rs1 + rs3w*z_rs3 + rs6w*z_rs6 + slopew*z_slope + accelw*z_accel
        df['Leader_Score'] = score
        return df[['RS1M','RS3M', 'RS6M','RS_Slope','RS_Acceleration','Leader_Score']]

    def final_ranking(self):
      """
       Return final ranking DataFrame. If breadth_df provided it will be joined.
       Adds a new column 'Rank' where highest Leader_Score gets rank 1.
      """
      # Compute the leader score
      merged = self.compute_leader_score()
      # Sort by Leader_Score descending
      merged = merged.sort_values('Leader_Score', ascending=False)

      # Add a new ranking column
      merged['Rank'] = merged['Leader_Score'].rank(method='first', ascending=False).astype(int)


      return merged

In [4]:
start_date = "2023-10-01"
end_date   = "2026-03-31"
week_end_date   = "2026-03-31"


tickers = ["XLF", "XLK", "XLV", "XLE", "XLY", "XLP", "XLI", "XLU", "XLRE","XLB","XLC","GBTC","GLD","MAGS","SPY"]
sectors = ["XLF", "XLK", "XLV", "XLE", "XLY", "XLP", "XLI", "XLU", "XLRE","XLB","XLC","GBTC","GLD","MAGS"]

if __name__ == "__main__":



  data = yf.download( tickers, start=start_date, end=end_date, interval="1d",auto_adjust=True)['Close']
  price_data = data.ffill().dropna()
  # Instantiate and run ----------
  ranker = STSectorRotationRanker(price_data=price_data, sector_etfs=sectors, benchmark='SPY')
  df_ranking = ranker.final_ranking()

  #print(breadth_df)
def label_rs1m(rs1m):
    if rs1m >= 0.06:  return "BULLISH ≥6%"
    if rs1m >= 0.02:  return "Positive +2–6%"
    if rs1m >= -0.02: return "Neutral ±2%"
    if rs1m >= -0.05: return "BEARISH –2 to –5%"
    return "COLLAPSING <–5%"

def label_rs3m(rs3m):
    if rs3m >= 0.08:  return "LEADING ≥8%"
    if rs3m >= 0.00:  return "Recovering 0–8%"
    if rs3m >= -0.05: return "Lagging –5 to 0%"
    return "DEAD <–5%"

def label_rs_slope(slope):
    if slope >= 0.0045:      return "ROCKET / Parabolic"
    if slope >= 0.0030:      return "Very Strong Uptrend"
    if slope >= 0.0020:      return "Strong Uptrend"
    if slope >= 0.0012:      return "Moderate Uptrend"
    if slope >= 0.0006:      return "Mild Uptrend"
    if slope >= 0.0000:      return "Flat / Barely Positive"
    if slope >= -0.0010:     return "Flat / Losing Ground"
    return "Downtrend — Avoid"
def interpret_acceleration(acc):
      if acc >= 0.00030: return "ROCKET / Exploding"
      if acc >= 0.00020: return "Very strong acceleration"
      if acc >= 0.00012: return "Strong acceleration"
      if acc >= 0.00008: return "Positive acceleration"
      if acc >= 0.00000: return "Neutral/slightly positive"
      return "Deceleration / avoid"

def regime(row):
    if row['Slope_Label'].startswith('ROCKET') or row['Accel_Description'].startswith('ROCKET'):
        return "MONSTER INCOMING"
    if 'Strong' in row['Slope_Label'] or 'Very Strong' in row['Accel_Description']:
        return "Clear Leadership"
    if row['RS_Slope'] > 0.0015 and row['RS_Acceleration'] > 0.00015:
        return "Strong & Accelerating"
    if row['RS_Slope'] > 0:
        return "Improving"
    return "Wait / Avoid"



df_ranking['RS1M_Label'] = df_ranking['RS1M'].apply(label_rs1m)
df_ranking['RS3M_Label'] = df_ranking['RS3M'].apply(label_rs3m)
df_ranking['Slope_Label'] = df_ranking['RS_Slope'].apply(label_rs_slope)
df_ranking['Accel_Description'] = df_ranking['RS_Acceleration'].apply(interpret_acceleration)
df_ranking['Regime'] = df_ranking.apply(regime, axis=1)

def quick_verdict(row):
    bad_short = row['RS1M'] < -0.02                     # short-term momentum against us
    bad_long  = row['RS3M'] < -0.08                     # still in deep bear market
    monster   = (row['RS_Slope'] >= 0.0035) or (row['RS_Acceleration'] >= 0.00025)

    if bad_short and not monster:
        return "AVOID — short-term momentum too negative"
    if bad_long and row['Rank'] <= 2:
        return "CAUTION — still deeply underwater"
    if row['RS1M'] >= 0.04 and row['RS_Slope'] >= 0.002:
        return "SCREAMING BUY"
    return "Good"

df_ranking['Quick_Verdict'] = df_ranking.apply(quick_verdict, axis=1)


df_ranking

[*********************100%***********************]  15 of 15 completed


,RS1M,RS3M,RS6M,RS_Slope,RS_Acceleration,Leader_Score,Rank,RS1M_Label,RS3M_Label,Slope_Label,Accel_Description,Regime,Quick_Verdict
ticker,,,,,,,,,,,,,
XLE,0.207284,0.537024,0.423945,0.000814,0.000256,2.759033,1,BULLISH ≥6%,LEADING ≥8%,Mild Uptrend,Very strong acceleration,Improving,Good
XLU,0.048753,0.177471,0.117653,0.000111,0.000040,0.568693,2,Positive +2–6%,LEADING ≥8%,Flat / Barely Positive,Neutral/slightly positive,Improving,Good
XLB,-0.000611,0.164899,0.161864,0.000003,0.000280,0.222480,3,Neutral ±2%,LEADING ≥8%,Flat / Barely Positive,Very strong acceleration,Improving,Good
XLP,-0.009712,0.146583,0.108643,-0.000064,0.000223,0.061009,4,Neutral ±2%,LEADING ≥8%,Flat / Losing Ground,Very strong acceleration,Wait / Avoid,Good
XLRE,-0.000438,0.088271,0.017501,-0.000065,-0.000029,-0.076297,5,Neutral ±2%,LEADING ≥8%,Flat / Losing Ground,Deceleration / avoid,Wait / Avoid,Good
XLF,0.023136,-0.047982,-0.056940,0.000065,0.000098,-0.227532,6,Positive +2–6%,Lagging –5 to 0%,Flat / Barely Positive,Positive acceleration,Improving,Good
GBTC,0.095437,-0.175988,-0.370962,0.000172,-0.000380,-0.264071,7,BULLISH ≥6%,DEAD <–5%,Flat / Barely Positive,Deceleration / avoid,Improving,Good
XLV,-0.024167,0.008093,0.114512,-0.000267,0.000214,-0.291678,8,BEARISH –2 to –5%,Recovering 0–8%,Flat / Losing Ground,Very strong acceleration,Wait / Avoid,AVOID — short-term momentum too negative
XLI,-0.040249,0.088231,0.075482,-0.000378,0.000179,-0.312268,9,BEARISH –2 to –5%,LEADING ≥8%,Flat / Losing Ground,Strong acceleration,Wait / Avoid,AVOID — short-term momentum too negative


# ⚙️ CONFIGURATION

In [5]:
WEIGHTS = {
    "RS": 0.30,
    "Momentum": 0.25,
    "Breadth": 0.20,
    "Trend": 0.15,
    "Efficiency": 0.10
}

LOOKBACKS = {
    "wtd": 5,
    "mtd": 21,
    "3m": 63
}


# 📥 DATA FETCHING

In [6]:
def fetch_prices(tickers, period="6mo"):
    data = yf.download(tickers, period=period, auto_adjust=True)["Close"]
    return data.dropna(axis=1, how="all")


# 📊 METRIC FUNCTIONS

In [7]:
def compute_returns(df, period):
    return df.pct_change(period).iloc[-1]

def compute_momentum(df):
    wtd = compute_returns(df, LOOKBACKS["wtd"])
    mtd = compute_returns(df, LOOKBACKS["mtd"])
    m3 = compute_returns(df, LOOKBACKS["3m"])
    return 0.2 * wtd + 0.4 * mtd + 0.4 * m3

def compute_relative_strength(df, benchmark):
    sector_returns = compute_returns(df, LOOKBACKS["3m"])
    benchmark_return = compute_returns(benchmark.to_frame(), LOOKBACKS["3m"]).iloc[0]
    return sector_returns / benchmark_return

def compute_trend_slope(df, window=20):
    ma = df.rolling(window).mean()
    return (ma.iloc[-1] - ma.iloc[-window]) / window

def compute_efficiency(df, window=20):
    returns = df.pct_change()
    vol = returns.rolling(window).std()
    total_return = df.pct_change(window).iloc[-1]
    return abs(total_return) / vol.iloc[-1]


# 📊 BREADTH (FROM YOUR INPUT)

In [8]:
def compute_breadth_from_percentages(breadth_dict):
    """
    Input format:
    {
        "XLK": {"ma20": 0.72, "ma50": 0.65},
        ...
    }
    """
    scores = {}

    for sector, values in breadth_dict.items():
        ma20 = values.get("ma20", np.nan)
        ma50 = values.get("ma50", np.nan)

        # Weighted breadth score
        scores[sector] = 0.6 * ma20 + 0.4 * ma50

    return pd.Series(scores)

# 🔄 NORMALIZATION

In [9]:
def normalize(series):
    if series.max() == series.min():
        return pd.Series(0.5, index=series.index)
    return (series - series.min()) / (series.max() - series.min())

# 🧠 SCORING ENGINE

In [10]:
def compute_scores(df_prices, benchmark_prices, breadth_series=None):

    momentum = compute_momentum(df_prices)
    rs = compute_relative_strength(df_prices, benchmark_prices)
    trend = compute_trend_slope(df_prices)
    efficiency = compute_efficiency(df_prices)

    df = pd.DataFrame({
        "RS": rs,
        "Momentum": momentum,
        "Trend": trend,
        "Efficiency": efficiency
    })

    # Add breadth if available
    if breadth_series is not None:
        df["Breadth"] = breadth_series

    # Normalize all metrics
    for col in df.columns:
        df[col] = normalize(df[col])

    # Select only available metrics
    active_weights = {k: v for k, v in WEIGHTS.items() if k in df.columns}

    # Re-normalize weights
    total_weight = sum(active_weights.values())
    active_weights = {k: v / total_weight for k, v in active_weights.items()}

    # Compute final score
    df["Score"] = 0
    for metric, weight in active_weights.items():
        df["Score"] += df[metric] * weight

    df["Rank"] = df["Score"].rank(ascending=False)

    return df.sort_values("Score", ascending=False)

# 🌍 MARKET CONFIGURATION

In [11]:
MARKETS = {
    "US": {
        "sectors": ["XLK", "XLC", "XLY", "XLRE", "XLF", "XLP","XLI", "XLU", "XLE", "XLB","XLV" ],
        "benchmark": "^GSPC"
    }

}

# ▶️ RUN ANALYSIS

In [12]:
def run_market_analysis(market_name, config, breadth_input=None):
    print(f"\n===== {market_name} =====")

    # Fetch prices
    sector_prices = fetch_prices(config["sectors"])
    benchmark_prices = fetch_prices([config["benchmark"]]).squeeze()

    # Handle optional breadth
    breadth_series = None
    if breadth_input is not None and market_name in breadth_input:
        breadth_series = compute_breadth_from_percentages(
            breadth_input[market_name]
        )

    # Compute scores
    results = compute_scores(sector_prices, benchmark_prices, breadth_series)

    #print(results)
    return results

# 🚀 EXECUTION

In [13]:
# Example Breadth Input (OPTIONAL)
breadth_input = {
    "US": {
        "XLY": {"ma20": 0.73, "ma50": 0.51},
        "XLP": {"ma20": 0.49, "ma50": 0.29},
        "XLE": {"ma20": 0.41, "ma50": 0.59},
        "XLF": {"ma20": 0.70, "ma50": 0.66},
        "XLV": {"ma20": 0.28, "ma50": 0.24},
        "XLI" :{"ma20": 0.65, "ma50": 0.52},
        "XLK": {"ma20": 0.77, "ma50": 0.74},
        "XLB": {"ma20": 0.46, "ma50": 0.46},
        "XLRE": {"ma20": 0.94, "ma50": 0.68},
        "XLC": {"ma20": 0.48, "ma50": 0.52},
        "XLU": {"ma20": 0.29, "ma50": 0.39},
    }
}

all_results = {}

for market, config in MARKETS.items():
    try:
        results = run_market_analysis(market, config, breadth_input)
        all_results[market] = results

        # Save results
        results.to_csv(f"{market}_sector_ranking.csv")

    except Exception as e:
        print(f"Error in {market}: {e}")

def combine_results(all_results):
    frames = []

    for market, df in all_results.items():
        temp = df.copy()
        temp["Market"] = market
        temp["Sector"] = temp.index
        frames.append(temp)

    combined_df = pd.concat(frames, axis=0)
    combined_df = combined_df.reset_index(drop=True)
    desired_order = [
        "Market",
        "Sector",
        "Global_Rank",
        "RS",
        "Momentum",
        "Trend",
        "Efficiency",
        "Breadth",
        "Score",
        "Rank"
    ]
        # keep only columns that actually exist
    cols = [c for c in desired_order if c in combined_df.columns]

    return combined_df[cols]

df = combine_results(all_results)
df
#df.to_csv("combined_results.csv")


===== US =====


[*********************100%***********************]  11 of 11 completed
[*********************100%***********************]  1 of 1 completed


,Market,Sector,RS,Momentum,Trend,Efficiency,Breadth,Score,Rank
0,US,XLK,0.535830,1.000000,1.000000,1.000000,0.863636,0.833476,1.0
1,US,XLRE,0.566674,0.618174,0.219664,0.446874,1.000000,0.602182,2.0
2,US,XLE,1.000000,0.615737,0.001049,0.106217,0.381119,0.540937,3.0
3,US,XLI,0.421922,0.513150,0.498765,0.225586,0.583916,0.469021,4.0
4,US,XLF,0.214901,0.355572,0.253767,0.350860,0.734266,0.373367,5.0
5,US,XLY,0.124070,0.362394,0.406522,0.332361,0.660839,0.354202,6.0
6,US,XLB,0.369812,0.352573,0.268044,0.102141,0.342657,0.318039,7.0
7,US,XLC,0.237419,0.335661,0.257326,0.277800,0.405594,0.302639,8.0
8,US,XLU,0.512827,0.379398,0.113749,0.000000,0.115385,0.288837,9.0
9,US,XLP,0.324018,0.294096,0.052853,0.044172,0.255245,0.234123,10.0


## Other markets

In [14]:
# =========================================
# DEFAULT NORMALIZATION
# =========================================

def normalize(series):
    if series.max() == series.min():
        return pd.Series(0.5, index=series.index)
    return (series - series.min()) / (series.max() - series.min())


# =========================================
# UNIVERSAL SCORING ENGINE
# =========================================

def compute_sector_score(df, weights, normalize_rs=True):
    """
    Universal scoring engine for TSX, ASX, NGX, etc.

    df: DataFrame with required features
    weights: dict of feature weights
    """

    df = df.copy()

    # --- Normalize RS if present ---
    if normalize_rs:
        for col in df.columns:
            if "RS" in col:
                df[col] = normalize(df[col])

    # --- Compute Score ---
    df["Score"] = 0

    for col, w in weights.items():
        if col in df.columns:
            df["Score"] += df[col] * w

    df["Rank"] = df["Score"].rank(ascending=False)

    return df.sort_values("Score", ascending=False)

🌍 MARKET CONFIGS (THIS IS THE REAL POWER)

In [15]:
# 🇨🇦 TSX CONFIG
TSX_WEIGHTS = {
    "RS_1M_angle": 0.15,  # early rotation signal
    "RS_3M_angle": 0.25,  # structuralleadership
    "RS_3M_above_zero": 0.2,
    "above_10W": 0.10,    # short trend alignment
    "above_30W": 0.15,    # regime support
    "price_angle_10W": 0.05,  # timing confirmation
    "price_angle_30W": 0.10   # regime filter (slow anchor)
}

# 🇦🇺 ASX CONFIG (slightly more momentum-sensitive)
ASX_WEIGHTS = {
    "RS_1M_angle": 0.20,  # early rotation signal
    "RS_3M_angle": 0.25,  # structuralleadership
    "RS_3M_above_zero": 0.15,
    "above_10W": 0.10,    # short trend alignment
    "above_30W": 0.10,    # regime support
    "price_angle_10W": 0.10,  # timing confirmation
    "price_angle_30W": 0.10   # regime filter (slow anchor)

}

# 🇳🇬 NGX CONFIG (your binary structure model)Slightly more weight on structure (because NGX is trend-driven)
NGX_WEIGHTS = {
    "RS_1M_angle": 0.25,  # early rotation signal
    "RS_3M_angle": 0.20,  # structuralleadership
    "RS_3M_above_zero": 0.10,
    "above_10W": 0.10,    # short trend alignment
    "above_30W": 0.10,    # regime support
    "price_angle_10W": 0.15,  # timing confirmation
    "price_angle_30W": 0.10   # regime filter (slow anchor)
}

SP500_WEIGHTS = {
    "RS_1M_angle": 0.15,        # still useful but not dominant
    "RS_3M_angle": 0.30,        # core leadership driver
    "RS_3M_above_zero": 0.15,   # regime confirmation (important in US)
    "above_10W": 0.10,          # short trend alignment
    "above_30W": 0.10,          # structural regime filter
    "price_angle_13W": 0.10,    # timing confirmation
    "price_angle_30W": 0.10     # macro trend anchor
}

def dict_to_sector_df(sector_dict):
    df = pd.DataFrame.from_dict(sector_dict, orient="index")
    df.index.name = "Sector"
    return df

## NGX Input Dictionary

In [16]:
ngx_input_dict = {
    "Consumer Goods": {
        "RS_1M_angle": 53.71, # MA length 5
        "RS_3M_angle": 38.47, # MA length 13
        "RS_3M_above_zero": 0,
        "above_10W": 1,
        "above_30W": 1,
        "price_angle_10W": 9.13,   # 10W SMA TIMING
        "price_angle_30W": 12.1    # 30W SMA regime confirmation
    },

    "Industrials": {
        "RS_1M_angle": 48.62,  # MA length 5
        "RS_3M_angle": 27.91,  # MA length 13
        "RS_3M_above_zero": 1,
        "above_10W": 1,
        "above_30W": 1,
        "price_angle_10W": 33.76,   # 10W SMA TIMING
        "price_angle_30W": 19.55    # 30W SMA regime confirmation
    },

    "Oil and Gas": {
        "RS_1M_angle": -56.45,  # MA length 5
        "RS_3M_angle": -39.77,  # MA length 13
        "RS_3M_above_zero": 1,
        "above_10W": 1,
        "above_30W": 1,
        "price_angle_10W": 42.83,   # 10W SMA TIMING
        "price_angle_30W": 25.01    # 30W SMA regime confirmation
    },

    "Banking": {
        "RS_1M_angle": 20.72,  # MA length 5
        "RS_3M_angle": 24.42,  # MA length 13
        "RS_3M_above_zero": 1,
        "above_10W": 1,
        "above_30W": 1,
        "price_angle_10W": 42.24,   # 10W SMA TIMING
        "price_angle_30W": 16.84    # 30W SMA regime confirmation
    },

     "Insurance": {
        "RS_1M_angle": -4.33,  # MA length 5
        "RS_3M_angle": -2.17,  # MA length 13
        "RS_3M_above_zero": 0,
        "above_10W": 0,
        "above_30W": 0,
        "price_angle_10W": -13.47,   # 10W SMA TIMING
        "price_angle_30W": -3.61    # 30W SMA regime confirmation
    }
}

# NGX ranking
df_ngx_input = dict_to_sector_df(ngx_input_dict)
#df_ngx_input
df_ngx = compute_sector_score(df_ngx_input, NGX_WEIGHTS)
df_ngx

,RS_1M_angle,RS_3M_angle,RS_3M_above_zero,above_10W,above_30W,price_angle_10W,price_angle_30W,Score,Rank
Sector,,,,,,,,,
Oil and Gas,0.000000,0.000000,1.0,1,1,42.83,25.01,9.225500,1.0
Banking,0.700527,0.820424,1.0,1,1,42.24,16.84,8.659216,2.0
Industrials,0.953794,0.865031,1.0,1,1,33.76,19.55,7.730455,3.0
Consumer Goods,1.000000,1.000000,0.0,1,1,9.13,12.10,3.229500,4.0
Insurance,0.473130,0.480573,0.0,0,0,-13.47,-3.61,-2.167103,5.0


## ASX/TSX Sector Inputs

In [17]:
market_config = {

    "SPY": {
        "benchmark": "^GSPC",
        "sectors": {
            "Financials": "XLF",
            "Energy": "XLE",
            "Materials": "XLB",
            "Industrials": "XLI",
            "Technology": "XLK",
            "Utilities": "XLU",
            "Health Care": "XLV",
            "Consumer Discretionary": "XLY",
            "Consumer Staples": "XLP",
            "Real Estate": "XLRE",
            "Biotechnology": "XBI",
            "Bitcoin": "GBTC",
            "Communication Services": "XLC",
            "Semi-Conductors": "SOXX",
            "Gold":"GLD",
            "Bonds": "TLT",
            "MAGS": "MAGS"
        }
    },


    "TSX": {
        "benchmark": "^GSPTSE",
        "sectors": {
            "Financials": "XFN.TO",
            "Energy": "XEG.TO",
            "Materials": "XMA.TO",
            "Industrials": "XGI.TO",
            "Technology": "XIT.TO",
            "Utilities": "XUT.TO",
            "Health Care": "XHC.TO",
            "Consumer Discretionary": "XCD.TO",
            "Consumer Staples": "XST.TO"
        }
    },

    "ASX": {
        "benchmark": "^AXJO",
        "sectors": {
            "Financials": "^AXFJ",
            "Materials": "^AXMJ",
            "Energy": "^AXEJ",
            "Industrials": "^AXNJ",
            "Technology": "^AXIJ",
            "Utilities": "^AXUJ",
            "Health Care": "^AXHJ",
            "Consumer Discretionary": "^AXDJ",
            "Consumer Staples": "^AXSJ",
            "Real Estate": "^AXRE",
            "Communication Services": "^AXTJ"
        }
    }
}

In [18]:
def get_data(ticker, period="3y"):
    df = yf.download(ticker, period=period, interval="1wk", auto_adjust=True)
    df = df.dropna()
    return df["Close"]


def normalize(series):
    if series.max() == series.min():
        return pd.Series(0, index=series.index)
    return (series - series.min()) / (series.max() - series.min())


def angle(series):
    """
    Simple proxy for trend angle = slope of rolling regression proxy
    """
    return series.diff().rolling(4).mean()


# -----------------------------
# Core feature builder
# -----------------------------

def build_sector_features(sector_price, benchmark_price):
    aligned = pd.concat([sector_price, benchmark_price], axis=1).dropna()
    aligned.columns = ["sector", "benchmark"]

    rs = aligned["sector"] / aligned["benchmark"]

    # RS angles
    rs_1m = rs.pct_change(4)
    rs_3m = rs.pct_change(13)

    rs_1m_angle = angle(rs_1m)
    rs_3m_angle = angle(rs_3m)

    # MA structure
    ma_10w = aligned["sector"].rolling(10).mean()
    ma_30w = aligned["sector"].rolling(30).mean()

    above_10w = (aligned["sector"] > ma_10w).astype(int)
    above_30w = (aligned["sector"] > ma_30w).astype(int)

    # Price trend angles
    price_angle_10w = angle(aligned["sector"].rolling(10).mean())
    price_angle_30w = angle(ma_30w)

    # RS regime
    rs_3m_above_zero = (rs_3m > 0).astype(int)

    latest = {
        "RS_1M_angle": rs_1m_angle.iloc[-1],
        "RS_3M_angle": rs_3m_angle.iloc[-1],
        "RS_3M_above_zero": rs_3m_above_zero.iloc[-1],
        "above_10W": int(above_10w.iloc[-1]),
        "above_30W": int(above_30w.iloc[-1]),
        "price_angle_10W": price_angle_10w.iloc[-1],
        "price_angle_30W": price_angle_30w.iloc[-1],
    }

    return latest


# -----------------------------
# Market engine
# -----------------------------

def build_market_inputs(market_config):
    benchmark = get_data(market_config["benchmark"])

    output = {}

    for sector, ticker in market_config["sectors"].items():
        sector_price = get_data(ticker)

        features = build_sector_features(sector_price, benchmark)
        output[sector] = features

    return output

In [19]:
usx_data = build_market_inputs(market_config["SPY"])
df_usx = pd.DataFrame.from_dict(usx_data, orient="index")
usx_result = compute_sector_score(df_usx, SP500_WEIGHTS)

usx_result

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,RS_1M_angle,RS_3M_angle,RS_3M_above_zero,above_10W,above_30W,price_angle_10W,price_angle_30W,Score,Rank
Semi-Conductors,1.000000,1.000000,1.0,1,1,7.646687,5.391050,1.339105,1.0
Technology,0.729971,0.884658,1.0,1,1,1.288804,0.514030,0.776296,2.0
Industrials,0.416872,0.405060,1.0,1,1,-0.016414,0.678382,0.601887,3.0
Biotechnology,0.057153,0.351866,1.0,1,1,0.809750,1.169926,0.581125,4.0
Real Estate,0.416171,0.492245,1.0,1,1,0.135705,0.087308,0.568830,5.0
MAGS,0.768889,0.727176,0.0,1,1,0.193250,0.062809,0.539767,6.0
Materials,0.172903,0.262254,1.0,1,1,0.014668,0.237273,0.478339,7.0
Consumer Discretionary,0.610122,0.588715,0.0,1,1,-0.075303,-0.061908,0.461942,8.0
Communication Services,0.441783,0.458157,0.0,1,1,-0.046345,-0.026659,0.401049,9.0
Utilities,0.154542,0.377043,1.0,0,1,0.169894,0.126855,0.398980,10.0


In [20]:
tsx_data = build_market_inputs(market_config["TSX"])
df_tsx = pd.DataFrame.from_dict(tsx_data, orient="index")
tsx_result = compute_sector_score(df_tsx, TSX_WEIGHTS)

tsx_result

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


,RS_1M_angle,RS_3M_angle,RS_3M_above_zero,above_10W,above_30W,price_angle_10W,price_angle_30W,Score,Rank
Financials,0.761062,0.713689,1.0,1,1,0.437133,0.412713,0.805709,1.0
Energy,0.000000,0.220262,1.0,1,1,0.349904,0.257839,0.548345,2.0
Industrials,0.683394,0.412888,0.0,1,1,0.012750,0.267668,0.483135,3.0
Utilities,0.129850,0.360819,1.0,0,1,0.197954,0.121046,0.481685,4.0
Technology,0.941690,1.000000,0.0,1,0,0.511750,-0.363667,0.480474,5.0
Consumer Discretionary,1.000000,0.404500,0.0,1,0,-0.205750,-0.063944,0.334443,6.0
Consumer Staples,0.190908,0.460282,0.0,0,0,-0.227545,0.119888,0.144318,7.0
Health Care,0.389692,0.353121,0.0,0,0,-0.490750,0.104055,0.132602,8.0
Materials,0.126139,0.000000,0.0,0,0,-0.094430,0.371514,0.051351,9.0


In [21]:
asx_data = build_market_inputs(market_config["ASX"])
df_asx = pd.DataFrame.from_dict(asx_data, orient="index")
#df_asx
asx_result = compute_sector_score(df_asx, ASX_WEIGHTS)

asx_result

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


,RS_1M_angle,RS_3M_angle,RS_3M_above_zero,above_10W,above_30W,price_angle_10W,price_angle_30W,Score,Rank
Energy,0.000000,0.000000,1.0,1,1,162.784985,80.657495,24.694248,1.0
Materials,0.470407,0.286771,1.0,0,1,31.022510,172.844189,20.802444,2.0
Consumer Staples,0.182927,0.267200,1.0,0,1,66.697485,21.710824,9.194216,3.0
Utilities,0.151100,0.374211,1.0,1,1,64.367529,16.525008,8.563026,4.0
Financials,0.384754,0.550694,1.0,0,1,16.482520,4.921663,2.605043,5.0
Communication Services,0.371949,0.546108,1.0,1,0,6.687500,-4.745834,0.655083,6.0
Technology,1.000000,1.000000,0.0,1,0,-2.682501,-41.310835,-3.849334,7.0
Real Estate,0.639747,0.709663,0.0,1,0,-26.177496,-21.173336,-4.329718,8.0
Industrials,0.451027,0.492586,0.0,0,0,-36.412488,-16.889994,-5.116896,9.0
Consumer Discretionary,0.427517,0.498851,0.0,0,0,-42.400000,-35.470001,-7.576784,10.0


# XLK Monthly Engine

In [39]:


# -----------------------------
# 1. Load Data
# -----------------------------
def get_data(ticker: str = "XLK", period: str = "2y") -> pd.DataFrame:
    df = yf.download(ticker, period=period, interval="1d", auto_adjust=True, progress=False)
    return df.dropna()


# -----------------------------
# 2. ATR
# -----------------------------
def compute_atr(df: pd.DataFrame, window: int = 14) -> pd.DataFrame:
    df = df.copy()
    high_low = df["High"] - df["Low"]
    high_close = np.abs(df["High"] - df["Close"].shift())
    low_close = np.abs(df["Low"] - df["Close"].shift())
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df["ATR"] = tr.rolling(window=window, min_periods=1).mean()
    return df


# -----------------------------
# 3. ATR-Adjusted ZigZag
# -----------------------------
def compute_zigzag_atr(df: pd.DataFrame, k: float = 1.5) -> pd.DataFrame:
    df = df.copy()
    prices = df["Close"].values
    atr = df["ATR"].values
    n = len(prices)

    if n < 10:
        df["pivot"] = 0
        df["zz_threshold"] = np.nan
        return df

    volatility_pct = atr / prices
    base_threshold = volatility_pct.mean() * k

    pivots = np.zeros(n)
    last_pivot_idx = 0
    last_pivot_price = prices[0]
    direction = 0  # 1 = up, -1 = down

    for i in range(1, n):
        change = (prices[i] - last_pivot_price) / last_pivot_price

        if direction == 0:
            if abs(change) >= base_threshold:
                direction = 1 if change > 0 else -1
                pivots[last_pivot_idx] = -direction
                last_pivot_idx = i
                last_pivot_price = prices[i]

        elif direction == 1:   # Looking for swing high
            if prices[i] > last_pivot_price:
                last_pivot_idx = i
                last_pivot_price = prices[i]
            elif (last_pivot_price - prices[i]) / last_pivot_price >= base_threshold:
                pivots[last_pivot_idx] = 1
                direction = -1
                last_pivot_idx = i
                last_pivot_price = prices[i]

        elif direction == -1:  # Looking for swing low
            if prices[i] < last_pivot_price:
                last_pivot_idx = i
                last_pivot_price = prices[i]
            elif (prices[i] - last_pivot_price) / last_pivot_price >= base_threshold:
                pivots[last_pivot_idx] = -1
                direction = 1
                last_pivot_idx = i
                last_pivot_price = prices[i]

    # Mark last tentative pivot
    if last_pivot_idx > 0:
        pivots[last_pivot_idx] = direction

    df["pivot"] = pivots
    df["zz_threshold"] = base_threshold
    return df


# -----------------------------
# 4. Swing Info
# -----------------------------
# -----------------------------
# 4. Swing Info - FIXED (Safe from Series error)
# -----------------------------
def get_swing_info(df: pd.DataFrame):
    if "pivot" not in df.columns or df["pivot"].abs().sum() == 0:
        return None, None

    pivot_df = df[df["pivot"] != 0].copy()

    if len(pivot_df) < 2:
        return None, None

    # Confirmed previous swing (last two pivots)
    last_two = pivot_df.iloc[-2:]
    start_idx = last_two.index[0]
    end_idx = last_two.index[1]

    # Use .iloc[0] to force scalar value and avoid Series
    start_price = float(df.loc[start_idx, "Close"].iloc[0] if isinstance(df.loc[start_idx, "Close"], pd.Series) else df.loc[start_idx, "Close"])
    end_price   = float(df.loc[end_idx, "Close"].iloc[0] if isinstance(df.loc[end_idx, "Close"], pd.Series) else df.loc[end_idx, "Close"])

    confirmed_swing = {
        "start": start_idx,
        "end": end_idx,
        "direction": "up" if end_price > start_price else "down",
        "start_price": start_price,
        "end_price": end_price
    }

    # Current developing leg
    last_pivot_idx = pivot_df.index[-1]
    current_price_raw = df["Close"].iloc[-1]
    last_pivot_price_raw = df.loc[last_pivot_idx, "Close"]

    current_price = float(current_price_raw.iloc[0] if isinstance(current_price_raw, pd.Series) else current_price_raw)
    last_pivot_price = float(last_pivot_price_raw.iloc[0] if isinstance(last_pivot_price_raw, pd.Series) else last_pivot_price_raw)

    current_leg = {
        "start": last_pivot_idx,
        "end": df.index[-1],
        "direction": "up" if current_price > last_pivot_price else "down",
        "start_price": last_pivot_price,
        "current_price": current_price
    }

    return confirmed_swing, current_leg

# -----------------------------
# 5. Metrics (Fixed)
# -----------------------------
# -----------------------------
# 5. Metrics - Safe version
# -----------------------------
def compute_metrics(df: pd.DataFrame, swing: dict) -> dict:
    if swing is None:
        return None

    start = swing["start"]
    price_start = swing["start_price"]
    price_end = swing.get("end_price") or swing.get("current_price")

    # Safe ATR mean
    atr_slice = df.loc[start:swing["end"], "ATR"]
    atr_avg = float(atr_slice.mean() if not atr_slice.empty else np.nan)

    move = abs(price_end - price_start)
    move_atr = move / atr_avg if atr_avg > 0 else np.nan

    duration = len(df.loc[start:swing["end"]])

    efficiency = move_atr / np.sqrt(max(duration, 1))

    return {
        "move": round(float(move), 2),
        "move_atr": round(float(move_atr), 2),
        "duration_bars": int(duration),
        "efficiency": round(float(efficiency), 4),
        "direction": swing["direction"],
        "atr_avg": round(float(atr_avg), 2)
    }


# -----------------------------
# 6. Time Symmetry (using bars)
# -----------------------------
def compute_time_symmetry(df: pd.DataFrame):
    pivot_df = df[df["pivot"] != 0]
    if len(pivot_df) < 3:
        return None

    last_three = pivot_df.index[-3:]
    t1 = len(df.loc[last_three[0]:last_three[1]]) - 1
    t2 = len(df.loc[last_three[1]:last_three[2]]) - 1

    ratio = round(t2 / t1, 3) if t1 != 0 else np.nan

    return {
        "leg1_bars": t1,
        "leg2_bars": t2,
        "time_ratio": ratio
    }


# -----------------------------
# 7. Classification
# -----------------------------
def classify_structure(prev_metrics: dict, curr_metrics: dict, symmetry: dict):
    if prev_metrics is None or curr_metrics is None:
        return {"signal": "Insufficient Data", "trend_strength": "N/A"}

    prev_eff = prev_metrics["efficiency"]
    curr_eff = curr_metrics["efficiency"]
    ratio = symmetry["time_ratio"] if symmetry else np.nan

    if curr_eff > prev_eff * 1.15:
        strength = "🟢 Strengthening (Momentum Increasing)"
    elif curr_eff < prev_eff * 0.85:
        strength = "🔴 Weakening (Exhaustion Building)"
    else:
        strength = "➖ Stable Momentum"

    if curr_eff > 0.18 and ratio < 1.1:
        signal = "Strong Bullish Structure"
    elif curr_eff > 0.12 and 0.8 < ratio < 1.3:
        signal = "Healthy Continuation"
    elif curr_eff < 0.07 or (ratio and ratio > 1.7):
        signal = "Weak/Exhausted Structure"
    else:
        signal = "Neutral"

    return {
        "signal": signal,
        "trend_strength": strength,
        "prev_efficiency": round(prev_eff, 4),
        "curr_efficiency": round(curr_eff, 4),
        "efficiency_change": round(curr_eff - prev_eff, 4)
    }


# -----------------------------
# 8. Main Runner
# -----------------------------
def run_analysis(ticker: str = "XLK", period: str = "2y", k: float = 1.5):
    df = get_data(ticker, period)
    df = compute_atr(df)
    df = compute_zigzag_atr(df, k=k)

    confirmed_swing, current_leg = get_swing_info(df)

    prev_metrics = compute_metrics(df, confirmed_swing)
    curr_metrics = compute_metrics(df, current_leg)
    symmetry = compute_time_symmetry(df)

    structure = classify_structure(prev_metrics, curr_metrics, symmetry)

    return {
        "ticker": ticker,
        "last_updated": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "zz_threshold": round(float(df["zz_threshold"].iloc[-1]), 4),
        "previous_swing": prev_metrics,
        "current_leg": curr_metrics,
        "time_symmetry": symmetry,
        "structure_analysis": structure
    }


# -----------------------------
# Execute
# -----------------------------
if __name__ == "__main__":
    result = run_analysis("XLK", period="2y", k=1.5)
    import pprint
    pprint.pprint(result, sort_dicts=False)

{'ticker': 'XLK',
 'last_updated': '2026-04-30 01:57',
 'zz_threshold': 0.0297,
 'previous_swing': {'move': 33.07,
                    'move_atr': 10.13,
                    'duration_bars': 20,
                    'efficiency': 2.2657,
                    'direction': 'up',
                    'atr_avg': 3.26},
 'current_leg': {'move': 1.46,
                 'move_atr': 0.52,
                 'duration_bars': 3,
                 'efficiency': 0.2992,
                 'direction': 'down',
                 'atr_avg': 2.82},
 'time_symmetry': {'leg1_bars': 23, 'leg2_bars': 19, 'time_ratio': 0.826},
 'structure_analysis': {'signal': 'Strong Bullish Structure',
                        'trend_strength': '🔴 Weakening (Exhaustion Building)',
                        'prev_efficiency': 2.2657,
                        'curr_efficiency': 0.2992,
                        'efficiency_change': -1.9665}}


In [40]:
result

{'ticker': 'XLK',
 'last_updated': '2026-04-30 01:57',
 'zz_threshold': 0.0297,
 'previous_swing': {'move': 33.07,
  'move_atr': 10.13,
  'duration_bars': 20,
  'efficiency': 2.2657,
  'direction': 'up',
  'atr_avg': 3.26},
 'current_leg': {'move': 1.46,
  'move_atr': 0.52,
  'duration_bars': 3,
  'efficiency': 0.2992,
  'direction': 'down',
  'atr_avg': 2.82},
 'time_symmetry': {'leg1_bars': 23, 'leg2_bars': 19, 'time_ratio': 0.826},
 'structure_analysis': {'signal': 'Strong Bullish Structure',
  'trend_strength': '🔴 Weakening (Exhaustion Building)',
  'prev_efficiency': 2.2657,
  'curr_efficiency': 0.2992,
  'efficiency_change': -1.9665}}